In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

In [ ]:
# Random obstacles.
radius = .1
centers = np.array([
    [.7, .3],
    [.8, .8],
    [.3, .2],
    [.3, .7],
    [.6, .6],
])

# Initial trajectory.
initial_points = np.vstack([
    np.linspace([0, 0], [.5, 0], 6),
    np.linspace([.5, .1], [.5, .7], 7),
    np.linspace([.6, .7], [1, .7], 5),
    np.linspace([1, .8], [1, 1], 3),
])

In [ ]:
# Objective function.
def objective_function(points):
    return cp.sum_squares(points[1:] - points[:-1])

# Linearization of constraints.
def linearized_constraint(p, p_new, c):
    diff = p - c
    dist = np.linalg.norm(diff)
    offset = radius - dist
    gradient = - diff / dist
    return offset + gradient @ (p_new - p) <= 0

# Difference of convex functions.
tol = 1e-3
solutions = [initial_points] # stores trajectories at all iterations for plotting
values = [objective_function(initial_points).value] # stores objective values
while True:
    points = solutions[-1] # previous trajectory
    new_points = cp.Variable(points.shape)
    constraints = [
        new_points[0] == points[0], # initial point is fixed
        new_points[-1] == points[-1] # final point is fixed
        ]
    for p, p_new in zip(points, new_points):
        for c in centers:
            # linearized obstacle avoidance
            constraints.append(linearized_constraint(p, p_new, c))
    cost = objective_function(new_points)
    prob = cp.Problem(cp.Minimize(cost), constraints)
    prob.solve()

    # Convergence check.
    if (values[-1] - prob.value) / prob.value < tol:
        break
    solutions.append(new_points.value)
    values.append(prob.value)

In [ ]:
# Plot result.
plt.figure()
for c in centers:
    patch = Circle(c, radius, facecolor='lightcoral', edgecolor='black')
    plt.gca().add_patch(patch)
for points, value in zip(solutions, values):
    rounded_value = np.round(value, 5)
    plt.plot(*points.T, marker='o', label=f"Objective value = {rounded_value}")
plt.legend()
plt.show()